In [15]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

def statement_to_dict_df(file, statement_name):
    # Extract the ticker from the filename by removing the statement name and file extension
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    metric_col = df.columns[0]

    rows = []

    # Iterate over each date column (starting from the second column) and create a dictionary of metrics for that date
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

In [16]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"
DATA_TICKER = DATA_DIR / "processed" / "layoffs_with_tickers.csv"
tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)
 
print(f"Found {len(tickers)} companies")

Found 2708 companies


In [ ]:
# master_rows = []

# for ticker in tickers:

#     try:
#         bs_file = BALANCE_SHEET_DIR / f"{ticker}_balancesheet.csv"
#         cf_file = CASH_FLOW_DIR / f"{ticker}_cashflow.csv"
#         fin_file = FINANCIALS_DIR / f"{ticker}_financials.csv"

#         bs = statement_to_dict_df(bs_file, "balancesheet")
#         cf = statement_to_dict_df(cf_file, "cashflow")
#         fin = statement_to_dict_df(fin_file, "financials")

#         company_df = (
#             bs.merge(cf, on=["company", "date"], how="outer")
#               .merge(fin, on=["company", "date"], how="outer")
#         )

#         master_rows.append(company_df)

#         print(f"Processed {ticker}")

#     except Exception as e:
#         # companies that are missing one of the statements will be skipped
#         print(f"Failed {ticker}: {e}")
for ticker in tickers:
    print(f"\nTicker: {ticker}")

    dfs = []

    for statement_name, directory, suffix in [
        ("balancesheet", BALANCE_SHEET_DIR, "_balancesheet.csv"),
        ("cashflow", CASH_FLOW_DIR, "_cashflow.csv"),
        ("financials", FINANCIALS_DIR, "_financials.csv"),
    ]:
        file = directory / f"{ticker}{suffix}"

        print(f"Checking: {file}")

        if file.exists():
            df = statement_to_dict_df(file, statement_name)

            print(f"  {statement_name}:")
            print(f"    shape = {df.shape}")
            print(f"    columns = {df.columns.tolist()}")

            if df.empty:
                print(f"    EMPTY DF for {file}")

            dfs.append(df)
        else:
            print(f"  {statement_name}: file missing")

    if len(dfs) == 0:
        continue

    company_df = dfs[0]

    for i, df in enumerate(dfs[1:], start=1):
        print(f"\nMerging with DF {i}")
        print("company_df columns:", company_df.columns.tolist())
        print("df columns:", df.columns.tolist())

        company_df = company_df.merge(
            df,
            on=["company", "date"],
            how="outer"
        )



Ticker: UBS
Checking: ..\data\balance_sheet\UBS_balancesheet.csv
  balancesheet:
    shape = (7, 3)
    columns = ['company', 'date', 'balancesheet']
Checking: ..\data\cashflows\UBS_cashflow.csv
  cashflow:
    shape = (6, 3)
    columns = ['company', 'date', 'cashflow']
Checking: ..\data\financials\UBS_financials.csv
  financials:
    shape = (7, 3)
    columns = ['company', 'date', 'financials']

Merging with DF 1
company_df columns: ['company', 'date', 'balancesheet']
df columns: ['company', 'date', 'cashflow']

Merging with DF 2
company_df columns: ['company', 'date', 'balancesheet', 'cashflow']
df columns: ['company', 'date', 'financials']

Ticker: MRK
Checking: ..\data\balance_sheet\MRK_balancesheet.csv
  balancesheet:
    shape = (7, 3)
    columns = ['company', 'date', 'balancesheet']
Checking: ..\data\cashflows\MRK_cashflow.csv
  cashflow:
    shape = (7, 3)
    columns = ['company', 'date', 'cashflow']
Checking: ..\data\financials\MRK_financials.csv
  financials:
    shape =

KeyError: 'company'

In [ ]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])

dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(
    dataset["balancesheet"]
).add_prefix("bs_")


cf_features = pd.json_normalize(
    dataset["cashflow"]
).add_prefix("cf_")

fin_features = pd.json_normalize(
    dataset["financials"]
).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)
# features_df.fillna("NaN")
# print(features_df.fillna("NaN").head())

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

# Verify
print(features_df.shape)
print(features_df.head())